# Evaluate complete conversations, not isolated turns

This credential-free lab adapts MLflow's [multi-turn agent cookbook](https://mlflow.org/cookbook/multi-turn-agent/) to the platform's session and release controls. Single-turn scores miss failures that emerge across a conversation: an unanswered follow-up, unresolved frustration, or policy drift after repeated requests.

Production state must live in the application or a durable framework store. A process-global conversation dictionary is acceptable only as a toy fixture and is not used here. Each real turn gets its own trace and shares one opaque, pseudonymous session ID.

In [ ]:
import sys
from pathlib import Path

repo_root = next(
    (
        path
        for path in (Path.cwd(), *Path.cwd().parents)
        if (path / "examples" / "support").is_dir()
    ),
    None,
)
if repo_root is None:
    raise FileNotFoundError("Open the cloned repository as your workspace.")
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

## 1. Review three synthetic sessions

The fixtures are deliberately small and deterministic. Every session is scoped to the same application release, environment, and evaluation batch so stale traces from another run cannot contaminate the result.

In [ ]:
from examples.support.agent_assurance import EVAL_BATCH, multi_turn_sessions

SESSIONS = multi_turn_sessions()
assert len({session["session_id"] for session in SESSIONS}) == len(SESSIONS)
assert all(session["eval_batch"] == EVAL_BATCH for session in SESSIONS)

## 2. Convert categorical session outcomes into explicit gate metrics

The connected MLflow scorers return categorical judgments. A release policy still needs named numeric metrics and critical-case rules. This offline layer makes that conversion visible and keeps deterministic policy checks independent of an LLM judge.

In [ ]:
from examples.support.agent_assurance import build_session_report

session_report = build_session_report(SESSIONS)
session_report

In [ ]:
from examples.support.agent_assurance import session_gate as evaluate_session_gate

session_metrics, session_policy, session_gate = evaluate_session_gate(session_report)
{
    "metrics": session_metrics,
    "gate_passed": session_gate.passed,
    "decision": "adopt" if session_gate.passed else "reject",
    "failures": [failure.model_dump(mode="json") for failure in session_gate.failures],
}

## 3. Optional connected MLflow conversational judges

The native `ConversationCompleteness`, `ConversationalGuidelines`, and `UserFrustration` scorers are experimental LLM judges. They evaluate **pre-collected traces** and do not accept a `predict_fn` for multi-turn evaluation.

For each real turn, open `mlflow.tracing.context(session_id=opaque_session_id)`, create exactly one traced agent invocation, and tag the trace with the evaluation batch, application release, and environment. Query all three tags before scoring. Do not attach a raw personal identifier.

In [ ]:
from examples.support.agent_assurance import run_native_conversational_judges

RUN_NATIVE_CONVERSATIONAL_JUDGES = False
SOURCE_TRACE_EXPERIMENT_ID = None
JUDGE_MODEL_URI = None  # Resolve the configured logical judge-model keylessly.
if RUN_NATIVE_CONVERSATIONAL_JUDGES:
    if not SOURCE_TRACE_EXPERIMENT_ID or not JUDGE_MODEL_URI:
        raise ValueError(
            "Set the source-trace experiment ID and governed judge model URI first"
        )
    print(
        run_native_conversational_judges(
            SESSIONS,
            source_trace_experiment_id=SOURCE_TRACE_EXPERIMENT_ID,
            judge_model_uri=JUDGE_MODEL_URI,
        )
    )
else:
    print("CONNECTED CONVERSATIONAL JUDGES SKIPPED")

## Result

The fixture is rejected because aggregate success cannot excuse one unresolved critical session. In a connected evaluation, preserve each judge rationale, fail on scorer errors, and keep these judges report-only until held-out human calibration meets the approved agreement threshold.